In [68]:
## 1. Importing Libraries
import math
import numpy as np
import pandas as pd

In [69]:
## 2. Loading and Cleaning the Data

DATA_URL = "https://www.football-data.co.uk/mmz4281/2223/E0.csv"

matches = pd.read_csv(DATA_URL)

matches = matches[["HomeTeam", "AwayTeam", "FTHG", "FTAG"]]

matches = matches.rename(
    columns={
        "HomeTeam": "home_team",
        "AwayTeam": "away_team",
        "FTHG": "home_goals",
        "FTAG": "away_goals",
    }
)

matches.head()

,home_team,away_team,home_goals,away_goals
0,Crystal Palace,Arsenal,0,2
1,Fulham,Liverpool,2,2
2,Bournemouth,Aston Villa,2,0
3,Leeds,Wolves,2,1
4,Newcastle,Nott'm Forest,2,0


In [70]:
## 3. Checking Available Teams

teams = sorted(matches["home_team"].unique())
teams

['Arsenal',
 'Aston Villa',
 'Bournemouth',
 'Brentford',
 'Brighton',
 'Chelsea',
 'Crystal Palace',
 'Everton',
 'Fulham',
 'Leeds',
 'Leicester',
 'Liverpool',
 'Man City',
 'Man United',
 'Newcastle',
 "Nott'm Forest",
 'Southampton',
 'Tottenham',
 'West Ham',
 'Wolves']

In [71]:
## 4. Selecting the Match

home_team = "Liverpool"
away_team = "Aston Villa"
max_goals = 10

market_odds = {
    "home_win": 1.57,
    "draw": 4.50,
    "away_win": 4.50,
}

In [72]:
## 5. Calculating Expected Goals

def calculate_expected_goals(matches, home_team, away_team):
    league_home_goals_avg = matches["home_goals"].mean()
    league_away_goals_avg = matches["away_goals"].mean()

    home_matches = matches[matches["home_team"] == home_team]
    away_matches = matches[matches["away_team"] == away_team]

    if home_matches.empty:
        raise ValueError(f"No home matches found for {home_team}.")

    if away_matches.empty:
        raise ValueError(f"No away matches found for {away_team}.")

    home_attack_strength = home_matches["home_goals"].mean() / league_home_goals_avg
    home_defense_factor = home_matches["away_goals"].mean() / league_away_goals_avg

    away_attack_strength = away_matches["away_goals"].mean() / league_away_goals_avg
    away_defense_factor = away_matches["home_goals"].mean() / league_home_goals_avg

    expected_home_goals = home_attack_strength * away_defense_factor * league_home_goals_avg
    expected_away_goals = away_attack_strength * home_defense_factor * league_away_goals_avg

    return {
        "home_team": home_team,
        "away_team": away_team,
        "league_home_goals_avg": league_home_goals_avg,
        "league_away_goals_avg": league_away_goals_avg,
        "home_attack_strength": home_attack_strength,
        "home_defense_factor": home_defense_factor,
        "away_attack_strength": away_attack_strength,
        "away_defense_factor": away_defense_factor,
        "expected_home_goals": expected_home_goals,
        "expected_away_goals": expected_away_goals,
    }

In [73]:
## 6. Expected Goals for the Selected Match

expected_goals = calculate_expected_goals(matches, home_team, away_team)

expected_goals_table = pd.DataFrame(
    [
        {"metric": "League home goals average", "value": expected_goals["league_home_goals_avg"]},
        {"metric": "League away goals average", "value": expected_goals["league_away_goals_avg"]},
        {"metric": f"{home_team} home attack strength", "value": expected_goals["home_attack_strength"]},
        {"metric": f"{home_team} home defense factor", "value": expected_goals["home_defense_factor"]},
        {"metric": f"{away_team} away attack strength", "value": expected_goals["away_attack_strength"]},
        {"metric": f"{away_team} away defense factor", "value": expected_goals["away_defense_factor"]},
        {"metric": f"Expected goals - {home_team}", "value": expected_goals["expected_home_goals"]},
        {"metric": f"Expected goals - {away_team}", "value": expected_goals["expected_away_goals"]},
    ]
)

expected_goals_table.round(3)

,metric,value
0,League home goals average,1.634
1,League away goals average,1.218
2,Liverpool home attack strength,1.481
3,Liverpool home defense factor,0.734
4,Aston Villa away attack strength,0.778
5,Aston Villa away defense factor,0.805
6,Expected goals - Liverpool,1.949
7,Expected goals - Aston Villa,0.696


In [74]:
## 7. Poisson Probability Function

def poisson_probability(goals, expected_goals):
    if goals < 0:
        raise ValueError("Goals must be greater than or equal to zero.")

    if expected_goals < 0:
        raise ValueError("Expected goals must be greater than or equal to zero.")

    return (
        math.exp(-expected_goals)
        * expected_goals ** goals
        / math.factorial(goals)
    )


def goal_probabilities(expected_goals, max_goals=10):
    probabilities = []

    for goals in range(max_goals + 1):
        probability = poisson_probability(goals, expected_goals)
        probabilities.append(probability)

    return pd.Series(
        probabilities,
        index=range(max_goals + 1),
        name="probability"
    )

In [75]:
## 8. Goal Probabilities

home_goal_probs = goal_probabilities(
    expected_goals=expected_goals["expected_home_goals"],
    max_goals=max_goals
)

away_goal_probs = goal_probabilities(
    expected_goals=expected_goals["expected_away_goals"],
    max_goals=max_goals
)

goal_probabilities_table = pd.DataFrame(
    {
        "goals": range(max_goals + 1),
        f"{home_team}_probability": home_goal_probs.values,
        f"{away_team}_probability": away_goal_probs.values,
    }
)

goal_probabilities_table.round(4)

,goals,Liverpool_probability,Aston Villa_probability
0,0,0.1424,0.4987
1,1,0.2775,0.3470
2,2,0.2705,0.1207
3,3,0.1758,0.0280
4,4,0.0857,0.0049
5,5,0.0334,0.0007
6,6,0.0108,0.0001
7,7,0.0030,0.0000
8,8,0.0007,0.0000
9,9,0.0002,0.0000


In [76]:
## 9. Scoreline Probability Matrix

def create_score_matrix(home_goal_probs, away_goal_probs):
    matrix = np.outer(home_goal_probs, away_goal_probs)

    score_matrix = pd.DataFrame(
        matrix,
        index=home_goal_probs.index,
        columns=away_goal_probs.index
    )

    score_matrix.index.name = "home_goals"
    score_matrix.columns.name = "away_goals"

    return score_matrix


score_matrix = create_score_matrix(home_goal_probs, away_goal_probs)

score_matrix.round(4)

away_goals,0,1,2,3,4,5,6,7,8,9,10
home_goals,,,,,,,,,,,
0,0.0710,0.0494,0.0172,0.0040,0.0007,0.0001,0.0,0.0,0.0,0.0,0.0
1,0.1384,0.0963,0.0335,0.0078,0.0014,0.0002,0.0,0.0,0.0,0.0,0.0
2,0.1349,0.0939,0.0326,0.0076,0.0013,0.0002,0.0,0.0,0.0,0.0,0.0
3,0.0877,0.0610,0.0212,0.0049,0.0009,0.0001,0.0,0.0,0.0,0.0,0.0
4,0.0427,0.0297,0.0103,0.0024,0.0004,0.0001,0.0,0.0,0.0,0.0,0.0
5,0.0167,0.0116,0.0040,0.0009,0.0002,0.0000,0.0,0.0,0.0,0.0,0.0
6,0.0054,0.0038,0.0013,0.0003,0.0001,0.0000,0.0,0.0,0.0,0.0,0.0
7,0.0015,0.0010,0.0004,0.0001,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0
8,0.0004,0.0003,0.0001,0.0000,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0


In [77]:
## 10. Match Outcome Probabilities
 
def calculate_outcome_probabilities(score_matrix):
    matrix = score_matrix.to_numpy()

    home_win = np.tril(matrix, k=-1).sum()
    draw = np.trace(matrix)
    away_win = np.triu(matrix, k=1).sum()

    total_probability = home_win + draw + away_win

    return {
        "home_win": home_win / total_probability,
        "draw": draw / total_probability,
        "away_win": away_win / total_probability,
    }


outcome_probabilities = calculate_outcome_probabilities(score_matrix)

outcome_probabilities_table = pd.DataFrame(
    [
        {"outcome": "home_win", "probability": outcome_probabilities["home_win"]},
        {"outcome": "draw", "probability": outcome_probabilities["draw"]},
        {"outcome": "away_win", "probability": outcome_probabilities["away_win"]},
    ]
)

outcome_probabilities_table["probability_%"] = outcome_probabilities_table["probability"] * 100

outcome_probabilities_table.round(3)

,outcome,probability,probability_%
0,home_win,0.670,67.035
1,draw,0.205,20.530
2,away_win,0.124,12.434


In [78]:
## 11. Fair Odds

def fair_odd(probability):
    if probability <= 0:
        return np.inf

    return 1 / probability


fair_odds = {
    outcome: fair_odd(probability)
    for outcome, probability in outcome_probabilities.items()
}

fair_odds_table = pd.DataFrame(
    [
        {"outcome": outcome, "fair_odd": odd}
        for outcome, odd in fair_odds.items()
    ]
)

fair_odds_table.round(2)

,outcome,fair_odd
0,home_win,1.49
1,draw,4.87
2,away_win,8.04


In [79]:
## 12. Market Odds
 
def compare_with_market_odds(outcome_probabilities, market_odds):
    rows = []

    for outcome, model_probability in outcome_probabilities.items():
        market_odd = market_odds[outcome]
        market_implied_probability = 1 / market_odd
        model_fair_odd = fair_odd(model_probability)
        expected_value = (model_probability * market_odd) - 1

        rows.append(
            {
                "outcome": outcome,
                "model_probability": model_probability,
                "model_probability_%": model_probability * 100,
                "fair_odd": model_fair_odd,
                "market_odd": market_odd,
                "market_implied_probability_%": market_implied_probability * 100,
                "expected_value_%": expected_value * 100,
                "has_value": expected_value > 0,
            }
        )

    return pd.DataFrame(rows)


odds_comparison = compare_with_market_odds(outcome_probabilities, market_odds)

odds_comparison.round(3)

,outcome,model_probability,model_probability_%,fair_odd,market_odd,market_implied_probability_%,expected_value_%,has_value
0,home_win,0.670,67.035,1.492,1.57,63.694,5.245,True
1,draw,0.205,20.530,4.871,4.50,22.222,-7.614,False
2,away_win,0.124,12.434,8.042,4.50,22.222,-44.045,False


In [80]:
## 13. Comparing Model Odds with Market Odds

print(f"Match: {home_team} vs {away_team}")

print("Expected goals")
print(f"{home_team}: {expected_goals['expected_home_goals']:.2f}")
print(f"{away_team}: {expected_goals['expected_away_goals']:.2f}")

print("Outcome probabilities")
for outcome, probability in outcome_probabilities.items():
    print(f"{outcome}: {probability * 100:.2f}%")

print("Fair odds")
for outcome, odd in fair_odds.items():
    print(f"{outcome}: {odd:.2f}")

print("Market comparison")
display(odds_comparison.round(3))

Match: Liverpool vs Aston Villa
Expected goals
Liverpool: 1.95
Aston Villa: 0.70
Outcome probabilities
home_win: 67.04%
draw: 20.53%
away_win: 12.43%
Fair odds
home_win: 1.49
draw: 4.87
away_win: 8.04
Market comparison


,outcome,model_probability,model_probability_%,fair_odd,market_odd,market_implied_probability_%,expected_value_%,has_value
0,home_win,0.670,67.035,1.492,1.57,63.694,5.245,True
1,draw,0.205,20.530,4.871,4.50,22.222,-7.614,False
2,away_win,0.124,12.434,8.042,4.50,22.222,-44.045,False


In [81]:
## 14. Potential Value Bets

value_bets = odds_comparison[odds_comparison["has_value"]].copy()
value_bets = value_bets.sort_values("expected_value_%", ascending=False)

value_bets.round(3)

,outcome,model_probability,model_probability_%,fair_odd,market_odd,market_implied_probability_%,expected_value_%,has_value
0,home_win,0.67,67.035,1.492,1.57,63.694,5.245,True


In [ ]:
import os

os.makedirs("data", exist_ok=True)

# 1. Expected goals summary
expected_goals_summary = pd.DataFrame({
    "team": [home_team, away_team],
    "team_type": ["home", "away"],
    "expected_goals": [
        expected_goals["expected_home_goals"],
        expected_goals["expected_away_goals"]
    ]
})

expected_goals_summary.to_csv(
    "data/expected_goals_summary.csv",
    index=False
)

# 2. Goal probabilities in long format for Power BI
goal_probabilities_powerbi = pd.concat(
    [
        pd.DataFrame({
            "team": [home_team] * (max_goals + 1),
            "team_type": ["home"] * (max_goals + 1),
            "goals": list(range(max_goals + 1)),
            "probability": home_goal_probs.values
        }),
        pd.DataFrame({
            "team": [away_team] * (max_goals + 1),
            "team_type": ["away"] * (max_goals + 1),
            "goals": list(range(max_goals + 1)),
            "probability": away_goal_probs.values
        })
    ],
    ignore_index=True
)

goal_probabilities_powerbi.to_csv(
    "data/goal_probabilities.csv",
    index=False
)

# 3. Score matrix in long format for Power BI
scorelines = []

for home_goals_count in range(max_goals + 1):
    for away_goals_count in range(max_goals + 1):
        scorelines.append({
            "home_team": home_team,
            "away_team": away_team,
            "home_goals": home_goals_count,
            "away_goals": away_goals_count,
            "scoreline": f"{home_goals_count}-{away_goals_count}",
            "probability": score_matrix.loc[home_goals_count, away_goals_count]
        })

score_matrix_long = pd.DataFrame(scorelines)

score_matrix_long.to_csv(
    "data/score_matrix.csv",
    index=False
)

# 4. Odds comparison
odds_comparison.to_csv(
    "data/odds_comparison.csv",
    index=False
)

print("Power BI CSV files generated successfully in the data/ folder.")

display(expected_goals_summary.round(3))
display(goal_probabilities_powerbi.head())
display(score_matrix_long.sort_values("probability", ascending=False).head(10))
display(odds_comparison.round(3))
 

Power BI CSV files generated successfully in the data/ folder.


,team,team_type,expected_goals
0,Liverpool,home,1.949
1,Aston Villa,away,0.696


,team,team_type,goals,probability
0,Liverpool,home,0,0.142371
1,Liverpool,home,1,0.277527
2,Liverpool,home,2,0.270494
3,Liverpool,home,3,0.175759
4,Liverpool,home,4,0.085653


,home_team,away_team,home_goals,away_goals,scoreline,probability
11,Liverpool,Aston Villa,1,0,1-0,0.138411
22,Liverpool,Aston Villa,2,0,2-0,0.134903
12,Liverpool,Aston Villa,1,1,1-1,0.096291
23,Liverpool,Aston Villa,2,1,2-1,0.093851
33,Liverpool,Aston Villa,3,0,3-0,0.087656
0,Liverpool,Aston Villa,0,0,0-0,0.071005
34,Liverpool,Aston Villa,3,1,3-1,0.060982
1,Liverpool,Aston Villa,0,1,0-1,0.049397
44,Liverpool,Aston Villa,4,0,4-0,0.042718
13,Liverpool,Aston Villa,1,2,1-2,0.033494


,outcome,model_probability,model_probability_%,fair_odd,market_odd,market_implied_probability_%,expected_value_%,has_value
0,home_win,0.670,67.035,1.492,1.57,63.694,5.245,True
1,draw,0.205,20.530,4.871,4.50,22.222,-7.614,False
2,away_win,0.124,12.434,8.042,4.50,22.222,-44.045,False
